# Classification Automatisée de l'Occupation des Sols (LULC)

## Introduction
La cartographie de l'occupation des sols (Land Use / Land Cover) est fondamentale pour l'aménagement du territoire, le suivi de l'urbanisation et la conservation. Ce notebook exploite l'IA **Prithvi EO** pour classer sans supervision les paysages du Congo en catégories majeures (Urbain, Forêt, Eau, etc.).

## Objectifs
*   **Zonage automatique** : Différencier les tissus urbains, forestiers et hydrologiques.
*   **Suivi de l'étalement** : Identifier les zones de pression anthropique.
*   **Base cartographique** : Créer un fond de carte thématique à jour.

## Méthodologie
1.  **Setup** : Installation de TerraTorch.
2.  **Acquisition** : Images Sentinel-2 (6 bandes spectrales).
3.  **Encodage sémantique** : Extraction des caractéristiques contextuelles globales par Prithvi.
4.  **Clustering K-Means** : Regroupement statistique des signatures pour former les classes d'occupation.

In [ ]:
# ====================================================
# ÉTAPE 1 : Setup
# ====================================================
!pip install geemap earthengine-api scikit-learn rasterio terratorch torch matplotlib seaborn -q

import ee, geemap, torch, rasterio
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from terratorch import BACKBONE_REGISTRY

try: ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print("✅ Système prêt")

## Zone d'Étude (ROI)
Alignement sur la zone régionale du Congo.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d\'étude')
Map

## Acquisition des Données Satellite
Prélèvement d'une image Sentinel-2 représentative.

In [ ]:
# ====================================================
# ÉTAPE 3 : Données Sentinel-2
# ====================================================
image = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(roi).median().clip(roi))

geemap.ee_export_image(image.select(['B2','B3','B4','B8','B11','B12']), 'land_cover.tif', scale=30, region=roi)

## Inférence et Classification K-Means
L'IA Prithvi extrait des dizaines de caractéristiques par pixel. L'algorithme K-Means les simplifie ensuite en 6 classes d'occupation terrestres distinctes.

In [ ]:
# ====================================================
# ÉTAPE 4 : Inférence et Segmentation
# ====================================================
model = BACKBONE_REGISTRY.build("prithvi_eo_v2_300", num_frames=1, in_chans=6, pretrained=True).eval().to('cpu')
with rasterio.open('land_cover.tif') as src: img = src.read().astype(np.float32) / 10000.0

with torch.no_grad():
    out = model(torch.from_numpy(img).unsqueeze(0))
    feats = out[0] if isinstance(out, list) else out
    
feats_np = feats[0, 1:].numpy()
kmeans = MiniBatchKMeans(n_clusters=6, n_init=3).fit(feats_np)
lc_labels = kmeans.labels_.reshape(int(np.sqrt(feats_np.shape[0])), -1)

plt.figure(figsize=(10, 8))
plt.imshow(lc_labels, cmap='tab10')
plt.colorbar(label='Classes d\'occupation des sols')
plt.title("Carte de l'Occupation des Sols par IA (LULC)")
plt.axis('off')
plt.show()